In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda:0',
    default_dtype="float64",
    model_dtype="float64",
    allow_tf32=False,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

import os
os.environ['NEQUIP_NUM_TASKS'] = '1'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./config/NaBr.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

In [3]:
config

{'_jit_bailout_depth': 2, '_jit_fusion_strategy': [('DYNAMIC', 3)], '_jit_fuser': 'fuser1', 'root': 'results/NaBr-tutorial', 'tensorboard': False, 'wandb': True, 'model_builders': ['allegro.model.Allegro', 'PerSpeciesRescale', 'ForceOutput', 'RescaleEnergyEtc'], 'dataset_statistics_stride': 1, 'device': 'cuda:0', 'default_dtype': 'float64', 'model_dtype': 'float64', 'allow_tf32': False, 'verbose': 'info', 'model_debug_mode': False, 'equivariance_test': False, 'grad_anomaly_mode': False, 'gpu_oom_offload': False, 'append': True, 'warn_unused': False, 'run_name': 'NaBr', 'seed': 123456, 'dataset_seed': 123456, 'r_max': 5.0, 'avg_num_neighbors': 'auto', 'BesselBasis_trainable': True, 'PolynomialCutoff_p': 6, 'l_max': 2, 'parity': 'o3_full', 'num_layers': 2, 'env_embed_multiplicity': 8, 'embed_initial_edge': True, 'two_body_latent_mlp_latent_dimensions': [32, 64, 128], 'two_body_latent_mlp_nonlinearity': 'silu', 'two_body_latent_mlp_initialization': 'uniform', 'latent_mlp_latent_dimensions

In [4]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3


trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

#config['model_input_fields'] = {'node_spin': o3.Irreps('1x1e')}

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

Torch device: cuda:0
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


Is ij diagonal
tensor(False)
Is ij diagonal
tensor(True)


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
    
from torch import nn
import math

data0 = AtomicData.to_AtomicDataDict(dataset[0])
#data0[AtomicDataDict.NODE_SPIN] = torch.randn_like(data0['pos'], device='cuda')

#data1 = with_edge_spin_length(data0, with_distance = True)

In [6]:
print(data0['pos'].shape)
print(data0['edge_index'].shape)
print(data0['cell'].shape)
print(data0['total_energy'].shape)
print(data0['forces'].shape)
print(data0['stress'].shape)
print(data0['free_energy'].shape)

torch.Size([64, 3])
torch.Size([2, 1146])
torch.Size([3, 3])
torch.Size([1])
torch.Size([64, 3])
torch.Size([3, 3])
torch.Size([1])


In [7]:
trainer.model = final_model

In [8]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

data_new = final_model(data0)

In [9]:
trainer.train()

Number of weights: 63784
Number of trainable weights: 63784
! Starting training ...

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      0     6        0.813        0.812      0.00135        0.223        0.309        0.697       0.0109


  Initialization     #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Initial Validation          0    3.802    0.002        0.894      0.00289        0.897        0.235        0.324        0.919       0.0144
Wall time: 3.804405066999607
! Best model        0    0.897

training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      1    10        0.515        0.499       0.0156        0.184        0.242         2.74       0.0428
      1    20        0.403        0.394      0.00826        0.162        0.215         1.99       0.0311
      1    30        0.21

      4    40       0.0495       0.0493     0.000171       0.0571       0.0761        0.287      0.00448
      4    50        0.058       0.0544      0.00361        0.063       0.0799         1.32       0.0206
      4    60       0.0301         0.03     6.45e-05        0.047       0.0593        0.176      0.00275
      4    70       0.0349       0.0322      0.00266       0.0495       0.0615         1.13       0.0177
      4    80       0.0368       0.0367      0.00012       0.0537       0.0656         0.24      0.00375
      4    90       0.0541       0.0474      0.00676       0.0593       0.0745          1.8       0.0282
      4   100       0.0711        0.071     6.99e-06       0.0643       0.0913        0.058     0.000906
      4   110       0.0247       0.0242     0.000557       0.0421       0.0533        0.517      0.00809
      4   120       0.0301       0.0262      0.00389       0.0441       0.0555         1.37       0.0214
      4   130       0.0465       0.0463     0.000251   

In [13]:
!/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-evaluate --train-dir results/NaBr-tutorial/NaBr --batch-size 1

Using device: cuda
Please note that _all_ machine learning models running on CUDA hardware are generally somewhat nondeterministic and that this can manifest in small, generally unimportant variation in the final test errors.
Loading model... 
[W418 14:39:58.768196369 init.cpp:809] Warning: nvfuser is no longer supported in torch script, use _jit_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:39:58.768231546 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:39:58.768243369 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
Is ij diagonal
tensor(False)
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-

In [18]:
!/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-deploy build --train-dir results/NaBr-tutorial/NaBr NaBr-deployed.pth

INFO:root:Loading best_model from training session...
[W418 14:44:44.123812794 init.cpp:809] Warning: nvfuser is no longer supported in torch script, use _jit_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:44:44.123839055 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:44:44.123849595 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
Is ij diagonal
tensor(False)
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/

In [17]:
config['type_names']

['Na', 'Br']